# Jupyter Notebook for Data Engineering and Analysis with PostgreSQL and Spark

This notebook demonstrates:
- Connecting to a PostgreSQL database using SQLAlchemy and Spark JDBC.
- Loading data from PostgreSQL into pandas and Spark DataFrames.
- Data type conversion between pandas and Spark.
- Basic data exploration and visualization setup.
- Example Spark jobs and DataFrame operations.

Libraries used:
- pandas, numpy, matplotlib, seaborn for data analysis and visualization.
- SQLAlchemy and psycopg2 for database connectivity.
- pyspark for distributed data processing.

Database: db_fraud (PostgreSQL)  
Author: [Your Name]  
Date: [Today's Date]

**Current Issues:**
- <span style="color:green">Potential connection failures to PostgreSQL (check credentials, network, or driver).</span>
- <span style="color:green">Data type mismatches when converting between pandas and Spark DataFrames.</span>
- <span style="color:green">Ensure the correct JDBC driver path and version for Spark connectivity.</span>
- <span style="color:green">Large data loads may require additional memory or Spark configuration.</span>
- <span style="color:red">Spark works in local mode, but cluster mode does not connect successfully (investigate cluster configuration, networking, and driver distribution).[PENDING]</span>

In [ ]:
from pyspark.sql import SparkSession

# Create a Spark session
spark = SparkSession.builder \
    .appName("SimpleSparkApp") \
    .master("local[*]") \
    .getOrCreate()

# Run a simple Spark job: create a DataFrame and show its contents
data = [("Alice", 34), ("Bob", 45), ("Cathy", 29)]
columns = ["Name", "Age"]
df = spark.createDataFrame(data, columns)
df.show()

# Stop the Spark session
spark.stop()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/10/13 15:20:16 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


+-----+---+
| Name|Age|
+-----+---+
|Alice| 34|
|  Bob| 45|
|Cathy| 29|
+-----+---+



In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import psycopg2
from sqlalchemy import create_engine
import warnings
from datetime import datetime, timedelta

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
warnings.filterwarnings('ignore')

# Configure display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

print("✅ Libraries imported successfully")
# Database connection parameters
# Note: Run the connection in the terminal first:
# psql -h 127.0.0.1 -p 5432 -U dfstechbi -d db_fraud
from urllib.parse import quote_plus
from sqlalchemy import text

DB_CONFIG = {
    'host': '10.205.161.118',
    'port': '5432',
    'database': 'db_fraud',
    'user': 'dfstechbi',
    'password': 'DfsTeChB1@923'  # You'll enter this when prompted
}

# Prompt for password
from getpass import getpass
DB_CONFIG['password'] = quote_plus(getpass('Enter database password: '))

# Create SQLAlchemy engine
connection_string = f"postgresql://{DB_CONFIG['user']}:{DB_CONFIG['password']}@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}"
engine = create_engine(connection_string)

# Test connection
try:
    with engine.connect() as conn:
        result = conn.execute(text("SELECT version();"))
        print("✅ Database connection successful!")
        print(f"PostgreSQL version: {result.fetchone()[0].split(',')[0]}")
except Exception as e:
    print(f"❌ Connection failed: {e}")

query = f"SELECT * FROM public.stixor_iar_jul_fraud_only_with_mbar"
df = pd.read_sql(query, engine)
df.head()

✅ Libraries imported successfully
✅ Database connection successful!
PostgreSQL version: PostgreSQL 17.6 on x86_64-pc-linux-gnu


In [50]:
query = f"SELECT * FROM public.stixor_iar_20250701_sample"
df = pd.read_sql(query, engine)
df.head()

,data_date,trans_id,trans_initiate_time,customer_msisdn,trx_channel,trx_type,trx_status,ac_from,ac_to,start_balance,trx_amt,end_balance,utility_company,bill_ref_number,fee,fed,reason_type,pur_of_remit,ec,merchant_id
0,2025-07-01,83857229624,2025-07-01 12:11:28,Qmd3D9+I+8pclaI8VAYzCA==,NEW_JC_APP,Transfer(C2C),Completed,J39ZmpPkp8/v4dM69tDirA==,Qmd3D9+I+8pclaI8VAYzCA==,41334.45,2000.00,43334.45,None,None,0.00,0.00,Customer Transfer to Customer via New JC APP,None,None,None
1,2025-07-01,83857229641,2025-07-01 12:11:28,ctxi60+/zNi3flSNLhrd5Q==,Payment Gateway,Online Payment,Completed,bnHhUCAIDNbIPsvQCUDa2w==,ctxi60+/zNi3flSNLhrd5Q==,5131.22,2000.00,3131.22,None,None,23.20,3.20,Online Payments for Jazz Mobile Account via PGW,None,None,None
2,2025-07-01,83857229644,2025-07-01 12:11:28,+Xk/jUiwGyCnVgwy2G92zg==,NEW_JC_APP,Transfer(C2C),Completed,4rccFAubRrFiobP23lcq3Q==,+Xk/jUiwGyCnVgwy2G92zg==,229096.20,5000.00,234096.20,None,None,0.00,0.00,Customer Transfer to Customer via New JC APP,None,None,None
3,2025-07-01,83857229697,2025-07-01 12:11:29,p16csxNsZ8N1/cp7yXmqYg==,Payment Gateway,Online Payment,Completed,C5xhNYbBbEGYDxfh6vRTBQ==,p16csxNsZ8N1/cp7yXmqYg==,1304.91,300.00,1004.91,None,None,3.48,0.48,Online Payments for Jazz Mobile Account via PGW,None,None,None
4,2025-07-01,83857229883,2025-07-01 12:11:30,cWX8h75n8J51OXyU8Dpv+w==,Payment Gateway,Online Payment,Completed,FQZZZQylOANwapxmURbH0Q==,cWX8h75n8J51OXyU8Dpv+w==,608.27,600.00,8.27,None,None,6.96,0.96,Online Payments for Jazz Mobile Account via PGW,None,None,None


In [52]:
import pyspark.sql.types as T

def pandas_dtype_to_spark(dtype):
    if dtype == 'int64':
        return T.LongType()
    elif dtype == 'float64':
        return T.DoubleType()
    elif dtype == 'datetime64[ns]':
        return T.TimestampType()
    elif dtype == 'object':
        return T.StringType()
    elif dtype == 'int32':
        return T.IntegerType()
    else:
        return T.StringType()

schema = T.StructType([
    T.StructField(col, pandas_dtype_to_spark(str(df[col].dtype)), True)
    for col in df.columns
])

spark_df = spark.createDataFrame(df, schema=schema)
spark_df.show()


Py4JJavaError: An error occurred while calling z:org.apache.spark.api.python.PythonRDD.readRDDFromFile.
: java.lang.OutOfMemoryError: Java heap space
	at org.apache.spark.api.java.JavaRDD$.readRDDFromInputStream(JavaRDD.scala:252)
	at org.apache.spark.api.java.JavaRDD$.readRDDFromFile(JavaRDD.scala:239)
	at org.apache.spark.api.python.PythonRDD$.readRDDFromFile(PythonRDD.scala:297)
	at org.apache.spark.api.python.PythonRDD.readRDDFromFile(PythonRDD.scala)
	at java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke0(Native Method)
	at java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke(NativeMethodAccessorImpl.java:75)
	at java.base/jdk.internal.reflect.DelegatingMethodAccessorImpl.invoke(DelegatingMethodAccessorImpl.java:52)
	at java.base/java.lang.reflect.Method.invoke(Method.java:580)
	at py4j.reflection.MethodInvoker.invoke(MethodInvoker.java:244)
	at py4j.reflection.ReflectionEngine.invoke(ReflectionEngine.java:374)
	at py4j.Gateway.invoke(Gateway.java:282)
	at py

In [ ]:
from pyspark.sql import SparkSession

# Provide path for PostgreSQL JDBC driver
# Example: "/path/to/postgresql-42.6.0.jar"
jdbc_driver_path = "postgresql-42.7.1.jar"

spark = SparkSession.builder \
    .appName("Test2") \
    .master("spark://localhost:7077") \
    .config("spark.jars", jdbc_driver_path) \
    .getOrCreate()


# Use Spark's JDBC capabilities to connect to PostgreSQL and load data into a Spark DataFrame
DB_CONFIG = {
    'host': '10.205.161.118',
    'port': '5432',
    'database': 'db_fraud',
    'user': 'dfstechbi',
    'password': 'DfsTeChB1@923'  # You'll enter this when prompted
}

jdbc_url = f"jdbc:postgresql://{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}"
properties = {
    "user": DB_CONFIG['user'],
    "password": DB_CONFIG['password'],
    "driver": "org.postgresql.Driver"
}

# Example: Read a table from PostgreSQL into Spark DataFrame


25/10/13 17:26:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [2]:
table_name = "public.fraud"
spark_df_pg = spark.read.jdbc(url=jdbc_url, table=table_name, properties=properties)
spark_df_pg.show()

25/10/13 17:26:43 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
25/10/13 17:26:58 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
25/10/13 17:27:13 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
25/10/13 17:27:28 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
ERROR:root:KeyboardInterrupt while sending command.                 (0 + 0) / 1]
Traceback (most recent call last):
  File "/root/miniconda3/envs/fraud/lib/python3.10/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/root/min

KeyboardInterrupt: 

25/10/13 17:27:43 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
25/10/13 17:27:58 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
25/10/13 17:28:13 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
25/10/13 17:28:28 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
25/10/13 17:28:43 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
25/10/13 17:28:58 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure th